# Task 2A - 05c Robustness and stress tests
Is the model fragile? We check (1) hyper-parameter sensitivity, (2) trend choice, (3) agreement of MAE / RMSE / WAPE, (4) dependence on a single festival year, and (5) the spread across plausible model variants for the real 10 weeks.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2a')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
from task2a_common import *
from task2a_features import *
from task2a_models import *
from task2a_validation import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (INTERIM, METRICS, FIGURES): d.mkdir(parents=True, exist_ok=True)
o, cal = load_orders(); cal2 = enrich_calendar(cal); panel = build_panel(o, cal2)


## 1. Hyper-parameter sensitivity (should be a plateau, not a spike)

In [2]:
P = PROD_EXTRA
mkp = lambda b, **kw: DailyRidgeGB(P, trend='none' if b == 'Style' else 'damped', **kw)     # production trend per brand
cfgs = {}
for a in (1e-3, 1e-2, 1e-1, 1.0): cfgs[f'ridge alpha={a}'] = dict(alpha=a)
for ph in (0.5, 0.8, 0.9, 0.95, 1.0): cfgs[f'trend phi={ph} (Fresh only; Style has no trend)'] = dict(phi=ph)
for d, it in ((2, 60), (3, 60), (4, 60), (3, 30), (3, 120)): cfgs[f'gb depth={d} iter={it}'] = dict(gb_kw=dict(max_depth=d, max_iter=it))
res = [backtest(panel, b, lambda b=b, kw=kw: mkp(b, **kw), name=n) for b in ('Fresh', 'Style') for n, kw in cfgs.items()]
hp = summarize(pd.concat(res)); hp.to_csv(METRICS / 'task2a_hyperparam_sweep.csv', index=False)
hp.pivot(index='model', columns='brand', values='WAPE').round(4)

brand,Fresh,Style
model,,
gb depth=2 iter=60,0.0317,0.0772
gb depth=3 iter=120,0.0319,0.0760
gb depth=3 iter=30,0.0320,0.0769
gb depth=3 iter=60,0.0318,0.0761
gb depth=4 iter=60,0.0318,0.0768
ridge alpha=0.001,0.0318,0.0761
ridge alpha=0.01,0.0318,0.0761
ridge alpha=0.1,0.0317,0.0769
ridge alpha=1.0,0.0315,0.0846


**Reading.** Fresh WAPE stays between 3.15% and 3.23% across ridge strength, trend damping and boosting size - a flat plateau, so the chosen defaults are not tuned to noise. Style behaves the same except for heavy ridge shrinkage (alpha = 1: 8.5% vs 7.6%), which flattens the festival effects.

## 2. Trend: none vs full vs damped (agreed selection rule)

In [3]:
tr = pd.concat([backtest(panel, b, lambda tr_=tr_: DailyRidgeGB(P, trend=tr_), name=f'trend={tr_}') for b in ('Fresh', 'Style') for tr_ in ('none', 'full', 'damped')])
summarize(tr).sort_values(['brand', 'WAPE'])

,model,brand,n,MAE,RMSE,WAPE,bias
2,trend=full,Fresh,680.0,23.4947,31.9218,0.0316,-0.0106
0,trend=damped,Fresh,680.0,23.6448,32.2244,0.0318,-0.0122
4,trend=none,Fresh,680.0,42.6114,51.9577,0.0574,-0.0538
5,trend=none,Style,680.0,8.5141,13.8343,0.0761,-0.0330
1,trend=damped,Style,680.0,8.6197,14.1119,0.0771,-0.0362
3,trend=full,Style,680.0,8.6331,14.1332,0.0772,-0.0365


**Selection rule (agreed): lowest error on comparable ten-week windows.**
- **Fresh:** no trend is far worse (WAPE 5.7% vs 3.2%); full trend (3.16%) and damped trend (3.18%) are a statistical tie, so we take the **damped** trend as the conservative choice for a 10-week extrapolation.
- **Style:** **no trend** wins on MAE, RMSE and WAPE (7.61% vs 7.71% damped), so Style is forecast without a trend.
- **Tech:** the damped trend reduces bias (-5.9% -> -3.4%) with a lower RMSE and a MAE within 0.2%, so it is kept.

## 3. Do MAE, RMSE and WAPE agree on the ranking?

In [4]:
sl = pd.read_csv(METRICS / 'task2a_ladder.csv')
for m in ('MAE', 'RMSE', 'WAPE'): sl[m + '_rank'] = sl.groupby('brand')[m].rank()
sl[['brand', 'model', 'MAE_rank', 'RMSE_rank', 'WAPE_rank']].sort_values(['brand', 'WAPE_rank'])

,brand,model,MAE_rank,RMSE_rank,WAPE_rank
10,Fresh,A5 + residual boosting (production Fresh),1.0,1.0,1.0
6,Fresh,A3 + a ramp per festival (per-depot),2.0,3.0,2.0
8,Fresh,A4 + pooled depots,3.0,2.0,3.0
4,Fresh,A2 + New Year ramp,4.0,4.0,4.0
2,Fresh,A1 + damped trend,5.0,5.0,5.0
0,Fresh,"A0 per-depot ridge, no trend",6.0,6.0,6.0
12,Style,A6 Style: drop the trend (production Style),1.0,1.0,1.0
11,Style,A5 + residual boosting (production Fresh),2.0,2.0,2.0
9,Style,A4 + pooled depots,3.0,3.0,3.0
7,Style,A3 + a ramp per festival (per-depot),4.0,4.0,4.0


The three metrics agree on the best model (production) and on the worst (no-trend ridge) for both brands. The only disagreements are between neighbouring rungs that differ by less than 0.1 percentage point (e.g. A3 vs A4 for Fresh), which are ties. So the chosen configuration does not depend on the unknown official metric.

## 4. How much does the forecast rely on a single New Year?

In [5]:
fops = future_ops(cal2, panel); cut = panel.date.max() + pd.Timedelta(days=1)
def forecast(trn_mask, brand='Fresh', make=None):
    m = (make or (lambda: make_model(brand)))().fit(panel[(panel.brand == brand) & trn_mask])
    f = pd.concat([fops.assign(depot=d, brand=brand) for d in DEPOTS]); f = f[f.iso_week.between(14, 23)]
    return f.assign(p=m.predict(f, cut)).groupby(['depot', 'iso_week']).p.sum()
ny24 = panel.date.between('2024-04-01', '2024-04-20'); ny25 = panel.date.between('2025-04-01', '2025-04-20')
out = pd.DataFrame({'all data': forecast(panel.date.notna()), 'without New Year 2024': forecast(~ny24), 'without New Year 2025': forecast(~ny25)})
out['spread %'] = ((out.max(axis=1) - out.min(axis=1)) / out['all data'] * 100).round(1)
out.loc[(slice(None), [15, 16]), :].round(1)

all data  without New Year 2024  without New Year 2025  spread %
depot      iso_week                                                                  
Kandy      15           745.2                  746.8                  739.8       0.9
           16           360.2                  359.5                  359.1       0.3
Peliyagoda 15          1412.6                 1415.8                 1401.8       1.0
           16           673.1                  671.3                  669.9       0.5

Dropping either New Year leaves the week 15/16 forecast within a modest range, because the model also learns from the other festivals and the ramp shape. This quantifies the main residual risk: *the New Year effect is estimated from two observed seasons.*

## 5. Spread across plausible model variants (real 10 weeks)

In [6]:
variants = {'production': lambda: make_model('Fresh'), 'per-depot ridge, NY ramp': lambda: DailyRidgeGB(('r_new_year',), pooled=False, gb=False),
            'pooled ridge, all ramps': lambda: DailyRidgeGB(PROD_EXTRA, gb=False), 'production, full trend': lambda: DailyRidgeGB(PROD_EXTRA, trend='full')}
fv = pd.DataFrame({n: forecast(panel.date.notna(), make=mk) for n, mk in variants.items()})
fv['max/min'] = (fv.max(axis=1) / fv.min(axis=1)).round(3); fv.round(1)

production  per-depot ridge, NY ramp  pooled ridge, all ramps  production, full trend  max/min
depot      iso_week                                                                                                
Kandy      14             538.5                     534.6                    533.3                   538.5      1.0
           15             745.2                     742.4                    741.6                   745.5      1.0
           16             360.2                     361.6                    360.7                   360.4      1.0
           17             561.8                     562.6                    561.0                   562.4      1.0
           18             516.3                     518.2                    515.3                   517.1      1.0
           19             518.9                     520.9                    519.4                   519.9      1.0
           20             523.7                     521.2                    519.6                   525.0      1.0
           21             531.7                     539.2                    530.7                   533.5      1.0
           22             618.4                     669.9                    617.6                   620.8      1.1
           23             517.2                     521.9                    520.3                   519.6      1.0
Peliyagoda 14            1024.3                    1019.3                   1014.1                  1024.4      1.0
           15            1412.6                    1415.3                   1405.7                  1413.0      1.0
           16             673.1                     673.2                    670.6                   673.5      1.0
           17            1066.5                    1063.3                   1064.4                  1067.5      1.0
           18             995.6                     984.5                    992.1                   997.0      1.0
           19             987.4                     990.5                    986.1                   989.2      1.0
           20             994.3                     991.0                    986.6                   996.6      1.0
           21            1009.2                    1020.4                   1007.3                  1012.2      1.0
           22            1175.7                    1258.2                   1172.5                  1179.9      1.1
           23             984.9                     992.2                    987.7                   989.1      1.0

Variants agree within about 1% in every week **except week 22** (Poson plus two paydays), where the model with a single shared ramp forecasts 8-10% higher. History explains why we trust the per-festival ramps: Poson's run-up is small (about +8% in 2025) whereas the New Year's is large, and the shared-ramp model cannot tell them apart. Week 22 remains the least certain week of the window.

## 6. Tested and rejected: catch-up demand around closures
Short weeks (fewer than 6 operating days) are under-forecast (see 05b). Hypothesis: when a day is closed, some demand moves onto the days that remain. We added `closed_wk` (closed Mon-Sat days in the ISO week) and `post_gap` (previous Mon-Sat day closed) as features.

In [7]:
P = PROD_EXTRA
rows = []
for b in ('Fresh', 'Style'):
    for n, ex in (('production', P), ('+ closed_wk', P + ('closed_wk',)), ('+ post_gap', P + ('post_gap',)), ('+ both', P + ('closed_wk', 'post_gap'))):
        rows.append(backtest(panel, b, lambda ex=ex, b=b: DailyRidgeGB(ex, trend='none' if b == 'Style' else 'damped'), name=n))
for n, cols in (('production', None), ('+ closed_wk', DOW + ['is_payday', 'festival_ramp', 'closed_wk']), ('+ both', DOW + ['is_payday', 'festival_ramp', 'closed_wk', 'post_gap'])):
    rows.append(backtest(panel, 'Tech', lambda cols=cols: TechRidge(cols=cols), name=n))
cu = pd.concat(rows); print(summarize(cu).sort_values(['brand', 'model']).to_string())
cu['short week'] = cu.n_days < 6
cu.groupby(['brand', 'model', 'short week']).apply(lambda g: pd.Series(metric_row(g.actual, g.pred))).round(3)[['WAPE', 'bias']].unstack('short week')

          model  brand      n      MAE     RMSE    WAPE    bias
0        + both  Fresh  680.0  24.1128  32.3053  0.0325 -0.0118
3   + closed_wk  Fresh  680.0  24.2421  32.6437  0.0326 -0.0119
6    + post_gap  Fresh  680.0  23.5206  31.8336  0.0317 -0.0123
8    production  Fresh  680.0  23.6448  32.2244  0.0318 -0.0122
1        + both  Style  680.0   8.4580  13.8411  0.0756 -0.0330
4   + closed_wk  Style  680.0   8.5351  13.9656  0.0763 -0.0330
7    + post_gap  Style  680.0   8.4792  13.8189  0.0758 -0.0326
9    production  Style  680.0   8.5141  13.8343  0.0761 -0.0330
2        + both   Tech  680.0   7.6636   9.3724  0.2939 -0.0349
5   + closed_wk   Tech  680.0   7.6620   9.3671  0.2939 -0.0348
10   production   Tech  680.0   7.6673   9.3555  0.2941 -0.0344


WAPE          bias       
short week         False  True   False  True 
brand model                                  
Fresh + both       0.031  0.054 -0.011 -0.020
      + closed_wk  0.031  0.057 -0.012 -0.019
      + post_gap   0.031  0.049 -0.011 -0.039
      production   0.031  0.053 -0.010 -0.047
Style + both       0.071  0.172 -0.028 -0.133
      + closed_wk  0.071  0.180 -0.028 -0.135
      + post_gap   0.071  0.169 -0.028 -0.119
      production   0.071  0.172 -0.028 -0.125
Tech  + both       0.285  0.424 -0.021 -0.235
      + closed_wk  0.285  0.423 -0.021 -0.232
      production   0.285  0.431 -0.018 -0.270

**Reading.** Overall error does not improve (changes of a few hundredths of a percentage point) and the short-week bias stays (Style about -12%, Tech about -25%). Only 40 short-week forecasts per brand exist, so we do **not** add features for them; instead the limitation is documented (short weeks 16 and 18 in the real window carry more uncertainty than the bands for ordinary weeks suggest).